# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [11]:
# Lane: Refresh / Content Opportunity Scoring


I'm choosing this lane because I already have early evidence it works. In Weeks 1-2 I built a hand-written rule (stale + visible pages) and compared it against a decision tree on the starter dataset. The tree beat the hand rule at Precision@50 while the hand rule won at Precision@20 — meaning a learned ranking adds real value once you look past the very top of the list, which is exactly the "ranked review queue" shape this lane produces. I want to spend the next 7 weeks turning that early signal into a properly validated, leakage-checked ranked queue with reason codes, rather than starting a new question from zero.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*


**Question:** Which pages should a content reviewer look at first for refresh, expansion, protection, pruning, or monitoring?

**Decision this improves:** Right now, deciding which of thousands of pages to review first is either manual or based on a simple hand rule. This project ranks pages so reviewers spend limited time on the most promising candidates first, instead of guessing or going alphabetically.

**Who acts, and what do they do:** A content reviewer or SEO editor with limited weekly capacity (e.g., can only review ~20-50 pages a week) uses the ranked queue and its reason codes to decide which pages to refresh, expand, protect, prune, or monitor.

**Cost of a wrong call:**
- **False positive** (flagged as worth reviewing, but wasn't really a problem): wastes a reviewer's limited time — the opportunity cost is a genuinely declining page that didn't get looked at instead.
- **False negative** (a real declining page never gets flagged): the page keeps losing visibility/traffic unnoticed until it's much harder to recover, or is caught too late.

Since reviewer time is the scarce resource, precision at the top of the list (Precision@20, Precision@50) matters more than overall accuracy — reviewers only ever see the top of the queue, never the whole dataset.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [12]:
import os

# Walk up until we find the repo root (identified by data/raw/ existing)
while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Still not found — check repo layout"
print("Found it.")

Working dir: c:\Users\PC\Desktop\Flyrank\FlyRank_ML_Internship_Starter
Found it.


In [13]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("Clients:", df["client_id"].nunique())

declining_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"Declining rate: {declining_rate:.3f}")

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale + visible pages (hand-rule candidates): {stale_visible} of {df.shape[0]}")

Rows: 30000 | Columns: 44
Clients: 32
Declining rate: 0.542
Stale + visible pages (hand-rule candidates): 17 of 30000


In [14]:
import json

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Baseline rule Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
print(f"Random forest Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
print(f"Improvement: {rf/base:.1f}x over the hand rule")

Baseline rule Precision@50: 0.240  (~12 of top 50 correct)
Random forest Precision@50: 0.680  (~34 of top 50 correct)
Improvement: 2.8x over the hand rule


**What these numbers show:**
- Out of 30,000 pages across 32 clients, 54.2% are currently labeled declining — a striking majority, meaning decline is the norm rather than the exception in this dataset, and there's a large candidate pool to prioritize among.
- Only 17 pages meet the simple "stale + visible" hand rule (days_since_last_update >= 180 AND impressions_90d >= 500) out of 30,000. This tiny overlap suggests staleness and visibility are somewhat inversely related in this data — pages that go 180+ days unrefreshed have often already lost much of their visibility by then, making the strict hand-rule combination too narrow to be a practical review queue on its own. The random forest beats the hand-written baseline by 2.8x at Precision@50 (0.680 vs 0.240 — roughly 34 of the top 50 flagged pages correct, versus 12 for the hand rule), using client-holdout validation. That's solid early evidence a learned ranking finds real signal beyond what a simple rule captures — worth spending the next 7 weeks validating properly on the fuller warehouse data, and worth investigating *why* the hand rule underperforms so much here (its 17-page pool is likely just too small and too strict to compete).

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed patterns in which signals associate with declining pages (visibility, staleness, age, position) on this anonymized starter slice.
- A ranked, decision-support queue — evidence-backed priority order, not guaranteed outcomes.
- Directional evidence that a learned model can outperform a simple hand rule at surfacing review candidates, measured by Precision@K under client-holdout validation.

**What I cannot claim:**
- That refreshing a flagged page will cause it to recover — that requires a real experiment (e.g., A/B test), which this data alone cannot provide.
- Anything about Google's actual ranking algorithm — I'm only measuring observed search/analytics signals, never reverse-engineering search behavior.
- That the label (`trend_direction == "down"`) is the ideal target — it's a current- window proxy, not a future outcome. A stronger version of this lane (which I plan to build toward) would predict decline in a future window from prior signals, avoiding the risk that today's "decline" bucket already bakes in some of what I'm trying to predict.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.